# QLoRA fine-tuning on Kaggle free GPU

DeepSeek-R1-Distill-Qwen-1.5B + 4-bit NF4 QLoRA + SFT, on a personal-profile QA dataset.

**Before you run this**

1. Settings -> Accelerator -> **GPU T4 x2** (or P100). Anything without CUDA will fail at the training step.
2. Settings -> Internet -> **On** (needed to pip-install and to download the base model).
3. Add-ons -> Secrets -> add `HF_TOKEN` **only if** you want to push the adapter to the Hub. Training works without it.
4. Upload your `personal_dataset.jsonl` as a **private Kaggle Dataset** and attach it, or paste it in the cell marked *Dataset*.

Runtime is roughly 25-45 minutes for 5 epochs on ~620 examples on a T4.

Nothing in this notebook prints raw personal answers.

## 1. Environment: install dependencies

In [ ]:
!pip install -q -U "transformers>=4.51.0" "trl>=0.17.0" "peft>=0.14.0" \
    "bitsandbytes>=0.45.0" "accelerate>=1.2.0" "datasets>=3.0.0" "huggingface_hub>=0.28.0"
print("installed")

## 2. Verify the GPU

Stop here if this cell reports no CUDA device - QLoRA cannot run on CPU.

In [ ]:
import os

# Kaggle's "GPU T4 x2" exposes two devices. Transformers' Trainer then wraps the
# model in DataParallel, which conflicts with device_map="auto" (the model is
# already sharded across devices) and fails at training time. A 1.5B model in
# 4-bit fits comfortably on one T4, so pin to a single GPU. This must happen
# BEFORE torch initialises CUDA, and it propagates to the `!python` subprocesses
# below because they inherit this kernel's environment.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"   # silences a fork warning

import torch

assert torch.cuda.is_available(), (
    "No CUDA GPU. Set Settings -> Accelerator -> GPU T4 x2 and restart the session."
)
name = torch.cuda.get_device_name(0)
total = torch.cuda.get_device_properties(0).total_memory / 1024**3
free, _ = torch.cuda.mem_get_info()
print(f"visible GPUs     : {torch.cuda.device_count()}  (pinned to 1)")
print(f"GPU              : {name}")
print(f"Total memory     : {total:.1f} GiB")
print(f"Free memory      : {free / 1024**3:.1f} GiB")
print(f"bfloat16 support : {torch.cuda.is_bf16_supported()}  (T4 -> False, falls back to fp16)")
print(f"torch            : {torch.__version__}")
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

## 3. Clone the repository

In [ ]:
import os, sys

REPO_URL = "https://github.com/Harikrishnan200/deepseek-finetuning.git"
WORKDIR  = "/kaggle/working/deepseek-finetuning"

if not os.path.exists(WORKDIR):
    !git clone --depth 1 $REPO_URL $WORKDIR
else:
    !cd $WORKDIR && git pull --ff-only

os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)
print("cwd:", os.getcwd())
!git rev-parse --short HEAD

## 4. Dataset

The personal dataset is **not** committed to the repo. Attach it as a private
Kaggle Dataset (Add Data -> your dataset). The cell below searches `/kaggle/input`
for it, copies it to `data/raw/personal_dataset.jsonl`, and prints the path and
record count only - never the contents.

In [ ]:
import shutil
from pathlib import Path

TARGET = Path("data/raw/personal_dataset.jsonl")
KAGGLE_INPUT = Path("/kaggle/input")

# Set this if auto-discovery fails; the listing below tells you what to use.
SOURCE = None

if SOURCE is None:
    if not KAGGLE_INPUT.exists():
        print("/kaggle/input does not exist - no dataset is attached to this notebook.")
        candidates = []
    else:
        # List EVERYTHING, not just .jsonl: a wrong extension is a common cause.
        everything = [p for p in KAGGLE_INPUT.rglob("*") if p.is_file()]
        print(f"{len(everything)} file(s) under /kaggle/input:")
        for p in everything[:40]:
            print(f"    {p}  ({p.stat().st_size:,} bytes)")
        if len(everything) > 40:
            print(f"    ... and {len(everything) - 40} more")

        # Accept any text-ish extension, preferring a 'personal' filename.
        candidates = [p for p in everything if p.suffix.lower() in {".jsonl", ".json", ".txt", ".csv"}]
        candidates.sort(key=lambda p: (0 if "personal" in p.name.lower() else 1,
                                       0 if p.suffix.lower() == ".jsonl" else 1))
    SOURCE = candidates[0] if candidates else None

if SOURCE and Path(SOURCE).exists():
    TARGET.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(SOURCE, TARGET)
    print(f"\ncopied {SOURCE} -> {TARGET}")
elif TARGET.exists():
    print(f"\nusing dataset already in the repo: {TARGET}")
else:
    raise FileNotFoundError(
        "No usable data file found under /kaggle/input.\n"
        "  1. Right panel -> Add Data -> attach your dataset, then re-run this cell.\n"
        "  2. If it IS attached, copy a path from the listing above into SOURCE.\n"
        "  3. If the listing is empty, the dataset finished uploading but was not "
        "attached to THIS notebook session - reattach and restart the kernel."
    )

# Sanity check the copy: first line must parse and carry the expected fields.
import json as _json
first = next(line for line in TARGET.open(encoding="utf-8") if line.strip())
record = _json.loads(first)
assert {"instruction", "response"} <= set(record), f"unexpected fields: {sorted(record)}"
print("records:", sum(1 for line in TARGET.open(encoding="utf-8") if line.strip()))
print("fields :", sorted(record))

In [ ]:
# Optional: the generalization set (reworded questions) is private by default, so
# it is not in the git clone. If you added generalization.jsonl to your Kaggle
# Dataset, copy it across; otherwise evaluation still runs and simply reports the
# generalization check as a warning instead of a score.
from pathlib import Path
import shutil

GEN_TARGET = Path("data/eval/generalization.jsonl")
found = [p for p in Path("/kaggle/input").rglob("*generalization*.jsonl")] if Path("/kaggle/input").exists() else []

if found:
    GEN_TARGET.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(found[0], GEN_TARGET)
    print(f"copied {found[0]} -> {GEN_TARGET}")
    print("generalization items:", sum(1 for line in GEN_TARGET.open() if line.strip()))
elif GEN_TARGET.exists():
    print(f"already present: {GEN_TARGET}")
else:
    print("No generalization.jsonl found - that evaluation will be skipped with a warning.")
    print("To include it, add data/eval/generalization.jsonl to your private Kaggle Dataset.")

## 5. Validate the dataset schema

In [ ]:
!python -u scripts/validate_dataset.py --strict

## 6. Deduplicate, detect leakage, and split

Records are grouped by instruction similarity *before* splitting, so reworded
variants of the same question cannot straddle train and test. `--fail-on-leakage`
stops the notebook if any held-out question also appears in training.

In [ ]:
!python -u scripts/prepare_dataset.py --config configs/qlora.yaml --fail-on-leakage

In [ ]:
import json

report = json.load(open("data/reports/leakage_report.json"))
print("split sizes    :", report["split"]["split_sizes"])
print("actual ratios  :", report["split"]["actual_ratios"])
print("groups         :", report["split"]["total_groups"])
print("max overlap    :", report["leakage"]["max_overlap_rate"])
print("leakage free   :", report["leakage"]["leakage_free"])

## 7. Inspect the model architecture

Confirm the LoRA target modules against the *real* module tree rather than
assuming names from a tutorial.

In [ ]:
from transformers import AutoConfig

from src.training.config import load_config

config = load_config("configs/qlora.yaml")
model_config = AutoConfig.from_pretrained(config.model_name)
print("architecture :", model_config.architectures)
print("model_type   :", model_config.model_type)
print("layers       :", model_config.num_hidden_layers)
print("hidden size  :", model_config.hidden_size)
print("attn heads   :", model_config.num_attention_heads,
      "| kv heads:", model_config.num_key_value_heads)

## 8. Train

Checkpoints, history, run metadata and the loss curve land in `artifacts/training/`.

In [ ]:
!python -u scripts/train_qlora.py --config configs/qlora.yaml

## 9. Training curve

In [ ]:
import json

from IPython.display import Image, display

history = json.load(open("artifacts/training/training_history.json"))["history"]
for entry in history:
    if "validation_loss" in entry:
        print(f"step {entry['step']:>5}  epoch {entry.get('epoch', 0):>5.2f}  "
              f"train {entry.get('train_loss', float('nan')):.4f}  "
              f"val {entry['validation_loss']:.4f}")

display(Image("artifacts/training/loss_curve.png"))

## 10. Full evaluation

Base vs fine-tuned on the untouched test set: perplexity, task metrics,
generalization, catastrophic forgetting, final leakage check, and the promotion gate.

This loads each model in turn (never both at once) to stay inside 16 GB.

In [ ]:
!python -u scripts/evaluate.py \
    --config configs/qlora.yaml \
    --eval-config configs/evaluation.yaml \
    --adapter-path artifacts/training/adapter

## 11. Read the report

In [ ]:
from IPython.display import Markdown, display

display(Markdown(open("artifacts/evaluation/final_report.md").read()))

In [ ]:
from pathlib import Path

from IPython.display import Image, display

for name in [
    "training_vs_validation_loss",
    "train_validation_gap",
    "perplexity_comparison",
    "base_vs_finetuned_task_score",
    "generalization_comparison",
    "catastrophic_forgetting_comparison",
]:
    path = Path(f"artifacts/evaluation/plots/{name}.png")
    if path.exists():
        display(Image(str(path)))
    else:
        # Absent by design when the matching evaluation was skipped.
        print(f"not generated: {name}")

## 12. Spot-check some answers side by side

In [ ]:
from pathlib import Path

# Prefer the reworded generalization set; fall back to the test split when it is
# not present (it is gitignored for privacy).
split = "data/eval/generalization.jsonl"
if not Path(split).exists():
    split = "data/processed/test.jsonl"
    print(f"generalization set unavailable - comparing on {split} instead\n")

!python -u scripts/compare_models.py \
    --config configs/qlora.yaml \
    --adapter-path artifacts/training/adapter \
    --from-split $split \
    --limit 8

## 13. Optionally publish to the Hugging Face Hub

Runs **only** if the gate verdict is `PASS`. Requires a Kaggle secret named
`HF_TOKEN` (Add-ons -> Secrets). The token is read into an environment variable
and never printed.

Only the adapter, tokenizer, model card, and aggregate evaluation reports are
uploaded. The personal dataset is never uploaded.

In [ ]:
import json
import os

HUB_MODEL_ID = "Harikrishnan200/deepseek-personal-qlora"

verdict = json.load(open("artifacts/evaluation/gate.json"))["verdict"]
print("gate verdict:", verdict)

if verdict != "PASS":
    print("Not publishing: the gate did not return PASS. Adjust configs/qlora.yaml and retrain.")
else:
    try:
        from kaggle_secrets import UserSecretsClient

        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
        print("HF_TOKEN loaded from Kaggle secrets")
    except Exception as exc:
        print("Could not load HF_TOKEN:", type(exc).__name__)

    if os.environ.get("HF_TOKEN"):
        !python -u scripts/push_to_hub.py \
            --config configs/qlora.yaml \
            --adapter-path artifacts/training/adapter \
            --hub-model-id $HUB_MODEL_ID \
            --require-pass
    else:
        print("HF_TOKEN not set - skipping publish.")

## 14. Download the artifacts

Kaggle keeps `/kaggle/working` after the session ends, but zip it up so you can
pull the adapter and reports down in one click from the notebook output panel.

In [ ]:
!cd /kaggle/working && zip -qr adapter_and_reports.zip \
    deepseek-finetuning/artifacts/training/adapter \
    deepseek-finetuning/artifacts/training/training_history.json \
    deepseek-finetuning/artifacts/training/run_metadata.json \
    deepseek-finetuning/artifacts/evaluation \
    deepseek-finetuning/data/reports
!ls -lh /kaggle/working/adapter_and_reports.zip

## Next experiment

Free GPU quota is limited (~30 GPU-hours/week), so run experiments **one at a time**
rather than sweeping. Swap the config and re-run from section 8:

```
!python scripts/train_qlora.py --config configs/experiments/exp_a_r8.yaml
!python scripts/train_qlora.py --config configs/experiments/exp_b_r16.yaml
!python scripts/train_qlora.py --config configs/experiments/exp_c_r32.yaml
```

Each experiment writes to its own `artifacts/experiments/<run_name>/` directory,
so runs never overwrite each other.